# Landmark Exploratory Analysis

Long-running exploratory landmark notebook. The original markdown headers (`# Análisis de Land marks`, `# Pytorch`, `# Keras`, `# PCA`, `# Importancia de Características`, etc.) are preserved as section dividers. Workflows covered: mediapipe-based landmark loading, interpolation to a common frame length, multiple classical ML baselines (logistic regression, KNN, SVM, RF), PyTorch and Keras LSTM experiments, PCA-based dimensionality reduction, and the signant-vs-interpreter classification study.

**Note:** the original code uses Windows-style backslash paths (`HANDS_LANDMARKS\LEFT_HAND_LANDMARKS`); on macOS/Linux these need to be replaced with forward slashes or `os.path.join` calls. The configuration cell below shows the cross-platform form.

## Configuration

In [ ]:
import os

# Cross-platform paths. Original notebook used Windows backslashes;
# adjust if your data lives elsewhere.
LEFT_HAND_DIR = os.path.join("HANDS_LANDMARKS", "LEFT_HAND_LANDMARKS")
RIGHT_HAND_DIR = os.path.join("HANDS_LANDMARKS", "RIGHT_HAND_LANDMARKS")
BODY_DIR = "BODY_LANDMARKS"
FACE_DIR = "FACE_LANDMARKS"


# Análisis de Land marks

In [ ]:
# se Importan las librerias necesarias

import os
import sys
import cv2
import pandas as pd
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

# Se cargan todos los archivos csv de la carpeta

os.listdir("HANDS_LANDMARKS\LEFT_HAND_LANDMARKS")

# Se crea una lista con los nombres de los archivos csv

files = os.listdir("HANDS_LANDMARKS\LEFT_HAND_LANDMARKS")

# Se cargan los archivos csv en una variable
Lhand = [pd.read_csv("HANDS_LANDMARKS\LEFT_HAND_LANDMARKS\\" + file) for file in files]
Rhand = [pd.read_csv("HANDS_LANDMARKS\RIGHT_HAND_LANDMARKS\\" + file) for file in files]
body = [pd.read_csv("BODY_LANDMARKS\\" + file) for file in files]
#face = [pd.read_csv("FACE_LANDMARKS\\" + file) for file in files]


In [ ]:
#se agrega una columna con el nombre del archivo
for i in range(len(Lhand)):

    body[i]["etiqueta"] = int(files[i][2:4])
    #face[i]["etiqueta"] = int(files[i][2:4])

In [ ]:
# Se interpolan los datos a partir de los datos existentes al tamaño de la cantidad de frames mas pequeña
max_frames = min([len(Lhand[i]) for i in range(len(Lhand))])
for i in range(len(Lhand)):
    Lhand[i] = Lhand[i].reindex(range(max_frames)).interpolate()
    Rhand[i] = Rhand[i].reindex(range(max_frames)).interpolate()
    body[i] = body[i].reindex(range(max_frames)).interpolate()
    #face[i] = face[i].reindex(range(max_frames)).interpolate()

In [ ]:
#se grafican los datos

import matplotlib.pyplot as plt

plt.plot(body[40]["BD_etiqueta"])

plt.show()



In [ ]:
borrar1=[]
for i in Lhand[0].columns:
    if "z" in i:
        borrar1.append(i)

borrar2=[]
#for i in face[0].columns:
    #if "z" in i:
     #   borrar2.append(i)
borrar1.append("Unnamed: 0")
borrar3=[]
for i in body[0].columns:
    if "z" in i:
        borrar3.append(i)
borrar3.append("Unnamed: 0")

In [ ]:
for i in range(len(Lhand)):
    for j in borrar1:
        Rhand[i].drop(columns=[j], inplace=True)
    borrar1=borrar1[0:21]
    for j in borrar1:
        Lhand[i].drop(columns=[j], inplace=True)
   # for j in borrar2:
    #    face[i].drop(columns=[j], inplace=True)
    for j in borrar3:
        body[i].drop(columns=[j], inplace=True)
        


In [ ]:
# Se Cambian los nombres de las columnas de los archivos csv

for i in range(len(Lhand)):
    Lhand[i].columns = [f"LH_{col}" for col in Lhand[i].columns]
    Rhand[i].columns = [f"RH_{col}" for col in Rhand[i].columns]
    body[i].columns = [f"BD_{col}" for col in body[i].columns]
   # face[i].columns = [f"FC_{col}" for col in face[i].columns]

In [ ]:
# Se concatenan los archivos csv en un solo dataframe

df2 = pd.concat([Lhand[i].join(Rhand[i]).join(body[i]) for i in range(len(Lhand))], ignore_index=True, axis=0)


In [ ]:
df2.drop(columns=["RH_Unnamed: 0"], inplace=True)
df2["LH_Unnamed: 0"] = df2["LH_Unnamed: 0"].astype(int)
# se cambia el nombre de la priomera columna

df2.rename(columns={"LH_Unnamed: 0": "frame"}, inplace=True)

In [ ]:
# Se guarda el dataframe en un archivo csv

df2.to_csv("df1.csv", index=False)

# CSV

In [ ]:
# Se Crea un modelo de clasificacion con la libreria Pytorch LSTM

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch as th
import pandas as pd

# Se carga el archivo csv


# Pytorch


In [ ]:

batch_size = 64
num_epochs = 10000
learning_rate = 0.01

# Se crea el modelo

train_dataset = MyDataset(X_train, y_train)
test_dataset = MyDataset(X_test, y_test)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

# Se entrena el modelo

total_step = len(train_loader)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyModel(nn.Module):
    def __init__(self, max_length_frames, output_length):
        super(MyModel, self).__init__()
        self.lstm1 = nn.LSTM(input_size=134, hidden_size=64, batch_first=True)
        self.dropout1 = nn.Dropout(0.5)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Softmax(dim=50)
        
        # Regularization
        self.l2_lambda = 0.001

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        x = F.relu(self.fc1(x[:, -1, :]))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)
        return x

    def l2_regularization(self):
        l2_reg = torch.tensor(0., requires_grad=True)
        for param in self.parameters():
            l2_reg += torch.norm(param, 2)
        return self.l2_lambda * l2_reg

# Ejemplo de uso
max_length_frames = 24  # Ejemplo de longitud máxima de las secuencias
output_length = 50  # Ejemplo de número de clases de salida
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = MyModel(max_length_frames, output_length).to(device)

# Definir el optimizador y la función de pérdida
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Ejemplo de paso de entrenamiento
def train_step(x, y):
    model.train()
    optimizer.zero_grad()
    outputs = model(x)
    loss = criterion(outputs, y) + model.l2_regularization()
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
#se entrena el modelo

for epoch in range(num_epochs):
    for i, (x, y) in enumerate(train_loader):
        x = x.to(device)
        y = y.to(device)
        
        # Forward pass
        outputs = model(x)
        loss = criterion(outputs, th.argmax(y, dim=1))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 5 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}], Loss: {loss.item()}')


In [ ]:
# Se evalua el modelo

model.eval()

with th.no_grad():
    correct = 0
    total = 0
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = th.argmax(labels, 1).to(device)
        outputs = model(sequences)
        _, predicted = th.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}")

# Keras

In [ ]:
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

early_stopping = EarlyStopping(monitor='accuracy', patience=10, restore_best_weights=True)

df2 = pd.read_csv("df2.csv")

# Se normalizan los datos

scaler = StandardScaler()
#df2.iloc[:, 1:151] = scaler.fit_transform(df2.iloc[:, 1:151])
df2.head()

# Se dividen los datos en caracteristicas y etiquetas


In [ ]:
df2.info()

In [ ]:

X = df2.iloc[:, 1:151].values
y = df2["BD_etiqueta"].values[:]

scaler = StandardScaler()
X = scaler.fit_transform(X)

#se hace one hot encoding
X = X.reshape(-1,134, 150)
y = to_categorical(y)
Y=[]
for i in range(0,len(y),134):
    Y.append(y[i])
Y = np.array(Y)

# Se dividen los datos en entrenamiento y prueba con un 80% de entrenamiento y 20% de prueba sin barajear de cada clase

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2,stratify=Y)



In [ ]:
Y.shape

In [ ]:
#usando Keras y tensorflow crearemos un modelo de clasificacion con una red de 2 capas LSTM 2 capas densas y una capa de salida densa softmax que recibe 24 frames de 150 caracteristicas cada uno y devuelve una etiqueta de 3 clases
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from keras.layers import Input, Reshape
def model():
    model = Sequential()
    model.add(keras.layers.InputLayer(shape=(134, 150)))
    model.add(LSTM(32, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(64, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(128))
    model.add(Dropout(0.5))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    aja=Dense(50, activation='softmax')
    model.add(aja)

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    return model


In [ ]:
import pydot
from tensorflow.keras.utils import plot_model
model1 = model()
plot_model(model1, to_file='model.png', show_shapes=True, show_layer_names=True)

In [ ]:
model1 = model()
# se entrena el modelo en la GPU
model1.fit(X_train, y_train.reshape(-1,50), validation_data=(X_test, y_test.reshape(-1,50)), epochs=1200, batch_size=800, shuffle=True)

In [ ]:
#se grafica la curva de aprendizaje
import matplotlib.pyplot as plt

plt.plot(model1.history.history['accuracy'])
plt.plot(model1.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()


In [ ]:
# ...existing code...

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Realiza predicciones con el modelo entrenado
y_pred = model1.predict(X_test)


In [ ]:
y_pred = np.argmax(y_pred, axis=1)
y_test = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.preprocessing import label_binarize

# Convert y_test to one-hot encoded format
y_test_one_hot = label_binarize(y_test, classes=np.arange(50))

# Calcula las métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=1)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=1)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=1)
auc = roc_auc_score(y_test_one_hot, label_binarize(y_pred, classes=np.arange(50)), multi_class='ovr')

# Imprime las métricas
print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-Score: {f1}')
print(f'AUC: {auc}')

In [ ]:
model1.evaluate(X_test, y_test.reshape(-1,50))

In [ ]:
# Se crea la matriz de confusion se muestra los falsos positivos y falsos negativos
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = model1.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)
Y3pruebaval = np.argmax(y_test, axis=1)
cm = confusion_matrix(Y3pruebaval, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión 80-20')
plt.xlabel('Clase predicha')
plt.ylabel('Class verdadera')
plt.show()

# Quitando cada voluntario


In [ ]:
X2 = X
Y2 = Y
X3 = X
Y3 = Y
X4 = X
Y4 = Y
X5 = X
Y5 = Y
X6 = X
Y6 = Y


X2prueba=np.array([])
Y2prueba=np.array([])
X3prueba=np.array([])
Y3prueba=np.array([])
X4prueba=np.array([])
Y4prueba=np.array([])
X5prueba=np.array([])
Y5prueba=np.array([])
X6prueba=np.array([])
Y6prueba=np.array([])

for i in range(1000-20,-1,-20):
    X2 = np.delete(X2, np.arange(i,i+4,1),0) 
    Y2 = np.delete(Y2, np.arange(i,i+4,1),0)
    X2prueba = np.append(X2prueba,X[i:i+4])
    Y2prueba = np.append(Y2prueba,Y[i:i+4])
X2prueba = X2prueba.reshape(-1,134,150)
Y2prueba = Y2prueba.reshape(-1,50)

for i in range(1000-16,-1,-20):
    X3 = np.delete(X3, np.arange(i,i+4,1),0) 
    Y3 = np.delete(Y3, np.arange(i,i+4,1),0)
    X3prueba = np.append(X3prueba,X[i:i+4])
    Y3prueba = np.append(Y3prueba,Y[i:i+4])
X3prueba = X3prueba.reshape(-1,134,150)
Y3prueba = Y3prueba.reshape(-1,50)

for i in range(1000-12,-1,-20):
    X4 = np.delete(X4, np.arange(i,i+4,1),0) 
    Y4 = np.delete(Y4, np.arange(i,i+4,1),0)
    X4prueba = np.append(X4prueba,X[i:i+4])
    Y4prueba = np.append(Y4prueba,Y[i:i+4])
X4prueba = X4prueba.reshape(-1,134,150)
Y4prueba = Y4prueba.reshape(-1,50)

for i in range(1000-8,-1,-20):
    X5 = np.delete(X5, np.arange(i,i+4,1),0) 
    Y5 = np.delete(Y5, np.arange(i,i+4,1),0)
    X5prueba = np.append(X5prueba,X[i:i+4])
    Y5prueba = np.append(Y5prueba,Y[i:i+4])
X5prueba = X5prueba.reshape(-1,134,150)
Y5prueba = Y5prueba.reshape(-1,50)

for i in range(1000-4,-1,-20):
    X6 = np.delete(X6, np.arange(i,i+4,1),0) 
    Y6 = np.delete(Y6, np.arange(i,i+4,1),0)
    X6prueba = np.append(X6prueba,X[i:i+4])
    Y6prueba = np.append(Y6prueba,Y[i:i+4])
X6prueba = X6prueba.reshape(-1,134,150)
Y6prueba = Y6prueba.reshape(-1,50)



In [ ]:
#se mezclan los datos

from sklearn.utils import shuffle
X2, Y2 = shuffle(X2, Y2)
X3, Y3 = shuffle(X3, Y3)
X4, Y4 = shuffle(X4, Y4)
X5, Y5 = shuffle(X5, Y5)
X6, Y6 = shuffle(X6, Y6)


In [ ]:


model2 = model()
model2.fit(X2, Y2.reshape(-1,50), validation_data=(X2prueba, Y2prueba.reshape(-1,50)), epochs=1300, shuffle=True, batch_size=800)
plt.plot(model2.history.history['accuracy'])
plt.plot(model2.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
print("accuracy: ",model2.evaluate(X2prueba, Y2prueba.reshape(-1,50))[1])
model2.save("modelLSTM2.keras")








model3 = model()
model3.fit(X3, Y3.reshape(-1,50), validation_data=(X3prueba, Y3prueba.reshape(-1,50)), epochs=1300, shuffle=True, batch_size=800)
plt.plot(model3.history.history['accuracy'])
plt.plot(model3.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

print("accuracy: ",model3.evaluate(X3prueba, Y3prueba.reshape(-1,50))[1])
model3.save("modelLSTM3.keras")


model4 = model()
model4.fit(X4, Y4.reshape(-1,50), validation_data=(X4prueba, Y4prueba.reshape(-1,50)), epochs=1300, shuffle=True, batch_size=800)
plt.plot(model4.history.history['accuracy'])
plt.plot(model4.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
print("accuracy: ",model4.evaluate(X4prueba, Y4prueba.reshape(-1,50))[1])
model4.save("modelLSTM4.keras")



model5 = model()
model5.fit(X5, Y5.reshape(-1,50), validation_data=(X5prueba, Y5prueba.reshape(-1,50)), epochs=1300, shuffle=True, batch_size=800)
plt.plot(model5.history.history['accuracy'])
plt.plot(model5.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
print("accuracy: ",model5.evaluate(X5prueba, Y5prueba.reshape(-1,50))[1])
model5.save("modelLSTM5.keras")



model6 = model()
model6.fit(X6, Y6.reshape(-1,50), validation_data=(X6prueba, Y6prueba.reshape(-1,50)), epochs=1300, shuffle=True, batch_size=800)
plt.plot(model6.history.history['accuracy'])
plt.plot(model6.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

print("accuracy: ",model6.evaluate(X6prueba, Y6prueba.reshape(-1,50))[1])

model6.save("modelLSTM6.keras")


In [ ]:
plt.plot(model2.history.history['accuracy'])
plt.plot(model2.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# Visualización de landmarcks


In [ ]:
# Se muestran los landmarks de Rhand[555] en una imagen en blanco
import numpy as np
import cv2
import pandas as pd
import mediapipe as mp
import time 

def Visualizador(landmarksr,landmarksl,landmarksb):
    img = np.ones((720, 1280, 3), np.uint8) * 255
    for i in range(0, len(landmarksr)):
        for j in range(0, 21):
            x = int(landmarksr.iloc[i, 1+j*3] * 1280)
            y = int(landmarksr.iloc[i, j*3+2] * 720)
            cv2.circle(img, (x, y), 5, (0, 0, 255), -1)
        for j in range(0, 21):
            x = int(landmarksl.iloc[i, 1+j*3] * 1280)
            y = int(landmarksl.iloc[i, j*3+2] * 720)
            cv2.circle(img, (x, y), 5, (255, 0, 0), -1)
        for j in range(0, 33):
            x = int(landmarksb.iloc[i, 1+j*3] * 1280)
            y = int(landmarksb.iloc[i, j*3+2] * 720)
            cv2.circle(img, (x, y), 5, (0, 0, 0), -1)
        cv2.imshow("Seña", img)
        cv2.waitKey(0)
        img = np.ones((720, 1280, 3), np.uint8) * 255
    cv2.waitKey(0)
    cv2.destroyAllWindows()
for i in range(0, 50):
    Visualizador(Rhand[i*20],Lhand[i*20],body[i*20])


# Matrices 

In [ ]:
# se importan los modelos entrenados

from keras.models import load_model
model2 = load_model("modelLSTM2.keras")
model3 = load_model("modelLSTM3.keras")
model4 = load_model("modelLSTM4.keras")
model5 = load_model("modelLSTM5.keras")
model6 = load_model("modelLSTM6.keras")

# se crea la matriz de confusion para cada modelo

y_pred2 = model2.predict(X2prueba)
y_pred2 = np.argmax(y_pred2, axis=1)
Y2pruebaval = np.argmax(Y2prueba, axis=1)
cm2 = confusion_matrix(Y2pruebaval, y_pred2)
plt.figure(figsize=(10, 10))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusion voluntario 1")
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred3 = model3.predict(X3prueba)
y_pred3 = np.argmax(y_pred3, axis=1)
Y3pruebaval = np.argmax(Y3prueba, axis=1)
cm3 = confusion_matrix(Y3pruebaval, y_pred3)
plt.figure(figsize=(10, 10))
sns.heatmap(cm3, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusion voluntario 2")
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred4 = model4.predict(X4prueba)
y_pred4 = np.argmax(y_pred4, axis=1)
Y4pruebaval = np.argmax(Y4prueba, axis=1)
cm4 = confusion_matrix(Y4pruebaval, y_pred4)
plt.figure(figsize=(10, 10))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusion voluntario 3")
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred5 = model5.predict(X5prueba)
y_pred5 = np.argmax(y_pred5, axis=1)
Y5pruebaval = np.argmax(Y5prueba, axis=1)
cm5 = confusion_matrix(Y5pruebaval, y_pred5)
plt.figure(figsize=(10, 10))
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusion voluntario 4")
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred6 = model6.predict(X6prueba)
y_pred6 = np.argmax(y_pred6, axis=1)
Y6pruebaval = np.argmax(Y6prueba, axis=1)
cm6 = confusion_matrix(Y6pruebaval, y_pred6)
plt.figure(figsize=(10, 10))
sns.heatmap(cm6, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusion voluntario 5")
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


# Extración de caracteristicas


In [ ]:
import tsfel

# Se extraen las caracteristicas de los datos

cfg = tsfel.get_features_by_domain()
cfg.keys()

In [ ]:
X.shape

In [ ]:
x = tsfel.time_series_features_extractor(cfg, X, fs=60)

In [ ]:
[print(x.columns[i]) for i in indices[0:9]]

In [ ]:
carac = pd.read_csv("dffeatures.csv")

In [ ]:
# Se dividen los datos en caracteristicas y etiquetas

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xf = carac.iloc[:, :-1].values
yf = carac["etiqueta"].values[:]

scaler = StandardScaler()
Xf = scaler.fit_transform(Xf)

Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)


In [ ]:
yf = np.zeros(len(yf), dtype=int)

for i in range(0, len(yf), 20):
    yf[i:i+8] = 1
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)


In [ ]:
Xf_train.shape

In [ ]:
X2 = Xf
Y2 = yf
X3 = Xf
Y3 = yf
X4 = Xf
Y4 = yf
X5 = Xf
Y5 = yf
X6 = Xf
Y6 = yf


X2prueba=np.array([])
Y2prueba=np.array([])
X3prueba=np.array([])
Y3prueba=np.array([])
X4prueba=np.array([])
Y4prueba=np.array([])
X5prueba=np.array([])
Y5prueba=np.array([])
X6prueba=np.array([])
Y6prueba=np.array([])

for i in range(1000-20,-1,-20):
    X2 = np.delete(X2, np.arange(i,i+4,1),0) 
    Y2 = np.delete(Y2, np.arange(i,i+4,1),0)
    X2prueba = np.append(X2prueba,Xf[i:i+4])
    Y2prueba = np.append(Y2prueba,yf[i:i+4])



for i in range(1000-16,-1,-20):
    X3 = np.delete(X3, np.arange(i,i+4,1),0) 
    Y3 = np.delete(Y3, np.arange(i,i+4,1),0)
    X3prueba = np.append(X3prueba,Xf[i:i+4])
    Y3prueba = np.append(Y3prueba,yf[i:i+4])



for i in range(1000-12,-1,-20):
    X4 = np.delete(X4, np.arange(i,i+4,1),0) 
    Y4 = np.delete(Y4, np.arange(i,i+4,1),0)
    X4prueba = np.append(X4prueba,Xf[i:i+4])
    Y4prueba = np.append(Y4prueba,yf[i:i+4])



for i in range(1000-8,-1,-20):
    X5 = np.delete(X5, np.arange(i,i+4,1),0) 
    Y5 = np.delete(Y5, np.arange(i,i+4,1),0)
    X5prueba = np.append(X5prueba,Xf[i:i+4])
    Y5prueba = np.append(Y5prueba,yf[i:i+4])



for i in range(1000-4,-1,-20):
    X6 = np.delete(X6, np.arange(i,i+4,1),0) 
    Y6 = np.delete(Y6, np.arange(i,i+4,1),0)
    X6prueba = np.append(X6prueba,Xf[i:i+4])
    Y6prueba = np.append(Y6prueba,yf[i:i+4])





In [ ]:
from sklearn.utils import shuffle
X2, Y2 = shuffle(X2, Y2)
X3, Y3 = shuffle(X3, Y3)
X4, Y4 = shuffle(X4, Y4)
X5, Y5 = shuffle(X5, Y5)
X6, Y6 = shuffle(X6, Y6)


In [ ]:
Y2prueba

In [ ]:
X2prueba = X2prueba.reshape(-1,23400)
Y2prueba = Y2prueba.reshape(-1,1)

X3prueba = X3prueba.reshape(-1,23400)
Y3prueba = Y3prueba.reshape(-1,1)

X4prueba = X4prueba.reshape(-1,23400)
Y4prueba = Y4prueba.reshape(-1,1)

X5prueba = X5prueba.reshape(-1,23400)
Y5prueba = Y5prueba.reshape(-1,1)

X6prueba = X6prueba.reshape(-1,23400)
Y6prueba = Y6prueba.reshape(-1,1)

# Metodos Clasicos


In [ ]:
# Se crean los modelos de clasificacion: SVC, kNN, Random Forest, Gradient Boosting, ensamble de modelos y realiza un grid search para encontrar los mejores hiperparametros.

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:

svc = SVC()
knn = KNeighborsClassifier()
rf = RandomForestClassifier()
gb = GradientBoostingClassifier()

params_svc = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'poly', 'sigmoid'],
    'max_iter': [10000]
}

params_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

params_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3]
}

params_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1],
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3],'n_iter_no_change':[5]
}
grid_svc = GridSearchCV(svc, params_svc, cv=5,verbose=3)
grid_knn = GridSearchCV(knn, params_knn, cv=5,verbose=3)
grid_rf = GridSearchCV(rf, params_rf, cv=5,verbose=3)
grid_gb = GridSearchCV(gb, params_gb, cv=5,verbose=3)


In [ ]:
grid_svc.fit(Xf_train, yf_train)
grid_svc.best_params_


In [ ]:
svm = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)


In [ ]:

grid_knn.fit(Xf_train, yf_train)
grid_knn.best_params_


In [ ]:
knn = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')

In [ ]:

grid_rf.fit(Xf_train, yf_train)
grid_rf.best_params_


In [ ]:
rf = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)

In [ ]:

grid_gb.fit(Xf_train, yf_train)
grid_gb.best_params_

In [ ]:
#se entrenan los mejores modelos

svm.fit(Xf_train, yf_train)
knn.fit(Xf_train, yf_train)
rf.fit(Xf_train, yf_train)
#se evaluan los modelos

print("SVM: ", svm.score(Xf_test, yf_test))
print("kNN: ", knn.score(Xf_test, yf_test))
print("Random Forest: ", rf.score(Xf_test, yf_test))


In [ ]:
#se crea la matriz de confusion para cada modelo

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = svm.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión SVM')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


In [ ]:
y_pred = knn.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión kNN')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred = rf.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión Random Forest')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

In [ ]:
rl2 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rl3 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rl4 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rl5 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rl6 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)

rl2.fit(X2, Y2)
rl3.fit(X3, Y3)
rl4.fit(X4, Y4)
rl5.fit(X5, Y5)
rl6.fit(X6, Y6)

In [ ]:
svm2 = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
svm2.fit(X2, Y2)
svm3 = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
svm3.fit(X3, Y3)
svm4 = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
svm4.fit(X4, Y4)
svm5 = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
svm5.fit(X5, Y5)
svm6 = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
svm6.fit(X6, Y6)


In [ ]:
svm2 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm2.fit(X2, Y2)
svm3 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm3.fit(X3, Y3)
svm4 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm4.fit(X4, Y4)
svm5 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm5.fit(X5, Y5)
svm6 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm6.fit(X6, Y6)


In [ ]:
knn2 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn2.fit(X2, Y2)
knn3 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn3.fit(X3, Y3)
knn4 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn4.fit(X4, Y4)
knn5 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn5.fit(X5, Y5)
knn6 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn6.fit(X6, Y6)

In [ ]:
rf2 = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
rf2.fit(X2, Y2)
rf3 = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
rf3.fit(X3, Y3)
rf4 = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
rf4.fit(X4, Y4)
rf5 = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
rf5.fit(X5, Y5)
rf6 = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
rf6.fit(X6, Y6)


In [ ]:
print("RF2: ", rf2.score(X2prueba, Y2prueba))
print("RF3: ", rf3.score(X3prueba, Y3prueba))
print("RF4: ", rf4.score(X4prueba, Y4prueba))
print("RF5: ", rf5.score(X5prueba, Y5prueba))
print("RF6: ", rf6.score(X6prueba, Y6prueba))


In [ ]:
# se evaluan los modelos

print("KNN2: ", knn2.score(X2prueba, Y2prueba))
print("KNN3: ", knn3.score(X3prueba, Y3prueba))
print("KNN4: ", knn4.score(X4prueba, Y4prueba))
print("KNN5: ", knn5.score(X5prueba, Y5prueba))
print("KNN6: ", knn6.score(X6prueba, Y6prueba))

In [ ]:
#se evalua el modelo

print("SVM2: ", svm2.score(X2prueba, Y2prueba))
print("SVM3: ", svm3.score(X3prueba, Y3prueba))
print("SVM4: ", svm4.score(X4prueba, Y4prueba))
print("SVM5: ", svm5.score(X5prueba, Y5prueba))
print("SVM6: ", svm6.score(X6prueba, Y6prueba))


In [ ]:
print("RL2: ", rl2.score(X2prueba, Y2prueba))
print("RL3: ", rl3.score(X3prueba, Y3prueba))
print("RL4: ", rl4.score(X4prueba, Y4prueba))
print("RL5: ", rl5.score(X5prueba, Y5prueba))
print("RL6: ", rl6.score(X6prueba, Y6prueba))

# Se quita una repeticion de cada sena

In [ ]:
Xs=Xf
Ys=yf
# Se quita una muestra cada 4 y se agrega a un arreglo de prueba
Xsprueba=np.array([])
Ysprueba=np.array([])
for i in range(999,-1,-4):
    Xs = np.delete(Xs, i,0)
    Ys = np.delete(Ys, i,0)
    Xsprueba = np.append(Xsprueba,Xf[i])
    Ysprueba = np.append(Ysprueba,yf[i])
Xsprueba = Xsprueba.reshape(-1,23400)
Ysprueba = Ysprueba.reshape(-1,1)

In [ ]:
Xs[1].shape

In [ ]:
from sklearn.utils import shuffle

Xs, Ys = shuffle(Xs, Ys)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


svm7 = SVC(C=0.1, gamma=0.0001, kernel='rbf', max_iter=10000)
svm7.fit(Xs, Ys)
print("SVM7: ", svm7.score(Xsprueba, Ysprueba))

knn7 = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
knn7.fit(Xs, Ys)
print("kNN7: ", knn7.score(Xsprueba, Ysprueba))

rf7 = RandomForestClassifier(max_depth=9, min_samples_leaf=2, min_samples_split=2, n_estimators=200,criterion='entropy')
rf7.fit(Xs, Ys)
print("Random Forest7: ", rf7.score(Xsprueba, Ysprueba))

dt7 = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')
dt7.fit(Xs, Ys)
print("Decision Tree7: ", dt7.score(Xsprueba, Ysprueba))

rl7 = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rl7.fit(Xs, Ys)
print("RL7: ", rl7.score(Xsprueba, Ysprueba))


In [ ]:
# grid search para encontrar los mejores hiperparametros de un arbol de decision

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier()
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3]
}
grid_dt = GridSearchCV(dt, params_dt, cv=5,verbose=3)
grid_dt.fit(Xf_train, yf_train)
grid_dt.best_params_



In [ ]:
# Se Grafican todas las matrices de confusion

y_pred = svm.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión SVM')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred = knn.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión kNN')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred = rf.predict(Xf_test)
cm = confusion_matrix(yf_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión Random Forest')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred1 = svm2.predict(X2prueba)
y_pred2 = svm3.predict(X3prueba)
y_pred3 = svm4.predict(X4prueba)
y_pred4 = svm5.predict(X5prueba)
y_pred5 = svm6.predict(X6prueba)
cm1 = confusion_matrix(Y2prueba, y_pred1)
cm2 = confusion_matrix(Y3prueba, y_pred2)
cm3 = confusion_matrix(Y4prueba, y_pred3)
cm4 = confusion_matrix(Y5prueba, y_pred4)
cm5 = confusion_matrix(Y6prueba, y_pred5)
plt.figure(figsize=(10, 10))
sns.heatmap(cm1+cm2+cm3+cm4+cm5, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión SVM omitiendo un voluntario (suma)')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred1 = knn2.predict(X2prueba)
y_pred2 = knn3.predict(X3prueba)
y_pred3 = knn4.predict(X4prueba)
y_pred4 = knn5.predict(X5prueba)
y_pred5 = knn6.predict(X6prueba)
cm1 = confusion_matrix(Y2prueba, y_pred1)
cm2 = confusion_matrix(Y3prueba, y_pred2)
cm3 = confusion_matrix(Y4prueba, y_pred3)
cm4 = confusion_matrix(Y5prueba, y_pred4)
cm5 = confusion_matrix(Y6prueba, y_pred5)
plt.figure(figsize=(10, 10))
sns.heatmap(cm1+cm2+cm3+cm4+cm5, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión kNN omitiendo un voluntario (suma)')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred1 = rf2.predict(X2prueba)
y_pred2 = rf3.predict(X3prueba)
y_pred3 = rf4.predict(X4prueba)
y_pred4 = rf5.predict(X5prueba)
y_pred5 = rf6.predict(X6prueba)
cm1 = confusion_matrix(Y2prueba, y_pred1)
cm2 = confusion_matrix(Y3prueba, y_pred2)
cm3 = confusion_matrix(Y4prueba, y_pred3)
cm4 = confusion_matrix(Y5prueba, y_pred4)
cm5 = confusion_matrix(Y6prueba, y_pred5)
plt.figure(figsize=(10, 10))
sns.heatmap(cm1+cm2+cm3+cm4+cm5, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión Random Forest omitiendo un voluntario (suma)')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


In [ ]:

y_pred = svm7.predict(Xsprueba)
cm = confusion_matrix(Ysprueba, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión SVM reservando una seña de cada voluntario')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred = knn7.predict(Xsprueba)
cm = confusion_matrix(Ysprueba, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión kNN reservando una seña de cada voluntario')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

y_pred = rf7.predict(Xsprueba)
cm = confusion_matrix(Ysprueba, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión Random Forest reservando una seña de cada voluntario')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


# LSTM reservando una seña de cada voluntario

In [ ]:
ajj = np.argmax(Yv, axis=1)
from collections import Counter
Counter(ajj).most_common()

In [ ]:
Xv = X
Yv = Y



Xvprueba=np.array([])
Yvprueba=np.array([])


for i in range(996,-1,-4):
    Xv = np.delete(Xv, i,0) 
    Yv = np.delete(Yv, i,0)
    Xvprueba = np.append(Xvprueba,X[i])
    Yvprueba = np.append(Yvprueba,Y[i])
Xvprueba = Xvprueba.reshape(-1,134,150)
Yvprueba = Yvprueba.reshape(-1,50)



In [ ]:
from sklearn.utils import shuffle
Xv, Yv = shuffle(Xv, Yv)

In [ ]:
modelo = model()
modelo.fit(Xv, Yv.reshape(-1,50), validation_data=(Xvprueba, Yvprueba.reshape(-1,50)), epochs=600, shuffle=True, batch_size=64)
plt.plot(modelo.history.history['accuracy'])
plt.plot(modelo.history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
print("accuracy: ",modelo.evaluate(Xvprueba, Yvprueba.reshape(-1,50))[1])


In [ ]:
y_pred = modelo.predict(Xvprueba)
y_pred = np.argmax(y_pred, axis=1)

Yvpruebaval = np.argmax(Yvprueba, axis=1)

In [ ]:
# Calcula las métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=1)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=1)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=1)
auc = roc_auc_score(y_test, y_pred, multi_class='ovr')

# Imprime las métricas
print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-Score: {f1}')
print(f'AUC: {auc}')

In [ ]:
# se crea la matriz de confusion para el modelo
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred = modelo.predict(Xvprueba)
y_pred = np.argmax(y_pred, axis=1)
Yvpruebaval = np.argmax(Yvprueba, axis=1)
cm = confusion_matrix(Yvpruebaval, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión LSTM reservando una seña de cada voluntario')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


In [ ]:
# Se forma un ensamble de modelos

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

voting_clf = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf)], voting='hard')

voting_clf.fit(Xf_train, yf_train)
y_pred = voting_clf.predict(Xf_test)

print("Ensamble: ", accuracy_score(yf_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

import matplotlib.pyplot as plt

# Predicción del ensamble
y_pred_ensamble = voting_clf.predict(Xf_test)

# Creación de la matriz de confusión
cm_ensamble = confusion_matrix(yf_test, y_pred_ensamble)

# Visualización de la matriz de confusión
plt.figure(figsize=(10, 10))
sns.heatmap(cm_ensamble, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión del Ensamble')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

In [ ]:
voting_clf2 = VotingClassifier(estimators=[('svm', svm2), ('knn', knn2), ('rf', rf2)], voting='hard')
voting_clf2.fit(X2, Y2)
y_pred2 = voting_clf2.predict(X2prueba)
voting_clf3 = VotingClassifier(estimators=[('svm', svm3), ('knn', knn3), ('rf', rf3)], voting='hard')
voting_clf3.fit(X3, Y3)
y_pred3 = voting_clf3.predict(X3prueba)
voting_clf4 = VotingClassifier(estimators=[('svm', svm4), ('knn', knn4), ('rf', rf4)], voting='hard')
voting_clf4.fit(X4, Y4)
y_pred4 = voting_clf4.predict(X4prueba)
voting_clf5 = VotingClassifier(estimators=[('svm', svm5), ('knn', knn5), ('rf', rf5)], voting='hard')
voting_clf5.fit(X5, Y5)
y_pred5 = voting_clf5.predict(X5prueba)
voting_clf6 = VotingClassifier(estimators=[('svm', svm6), ('knn', knn6), ('rf', rf6)], voting='hard')
voting_clf6.fit(X6, Y6)
y_pred6 = voting_clf6.predict(X6prueba)

y_pred_ensamble2 = voting_clf2.predict(X2prueba)
y_pred_ensamble3 = voting_clf3.predict(X3prueba)
y_pred_ensamble4 = voting_clf4.predict(X4prueba)
y_pred_ensamble5 = voting_clf5.predict(X5prueba)
y_pred_ensamble6 = voting_clf6.predict(X6prueba)

cm1 = confusion_matrix(Y2prueba, y_pred_ensamble2)
cm2 = confusion_matrix(Y3prueba, y_pred_ensamble3)
cm3 = confusion_matrix(Y4prueba, y_pred_ensamble4)
cm4 = confusion_matrix(Y5prueba, y_pred_ensamble5)
cm5 = confusion_matrix(Y6prueba, y_pred_ensamble6)
plt.figure(figsize=(10, 10))
sns.heatmap(cm1+cm2+cm3+cm4+cm5, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión del Ensamble omitiendo un voluntario (suma)')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

#presicion de los ensambles

print("Ensamble2: ", accuracy_score(Y2prueba, y_pred_ensamble2))
print("Ensamble3: ", accuracy_score(Y3prueba, y_pred_ensamble3))
print("Ensamble4: ", accuracy_score(Y4prueba, y_pred_ensamble4))
print("Ensamble5: ", accuracy_score(Y5prueba, y_pred_ensamble5))
print("Ensamble6: ", accuracy_score(Y6prueba, y_pred_ensamble6))



In [ ]:
# Se forma un ensamble de modelos

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

voting_clf = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf)], voting='hard')

voting_clf.fit(Xs, Ys)
y_pred = voting_clf.predict(Xsprueba)

print("Ensamble: ", accuracy_score(Ysprueba, y_pred))

#matriz de confusion del ensamble

y_pred_ensamble = voting_clf.predict(Xsprueba)
cm_ensamble = confusion_matrix(Ysprueba, y_pred_ensamble)
plt.figure(figsize=(10, 10))
sns.heatmap(cm_ensamble, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión del Ensamble reservando una seña de cada voluntario')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


# Agrupación y Clasificación Intérprete - Sordo

In [ ]:
import numpy as np
import pandas as pd
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt


# Se importan los datos de los landmarks
df = pd.read_csv("df2.csv")
df.head()
X = df.iloc[:, 1:151].values[:]
y = carac["etiqueta"].values[:]
scaler = StandardScaler()
X = X.reshape(1000,-1, 150)
carac = pd.read_csv("dffeatures.csv")
carac.head()
yf = np.zeros(len(y), dtype=int)

for i in range(0, len(yf), 20):
    yf[i:i+8] = 1

    


In [ ]:
# Se dividen los datos en caracteristicas y etiquetas
Xf = carac.iloc[:, :-1].values
yf = carac["etiqueta"].values[:]
scaler2 = StandardScaler()
Xf = scaler2.fit_transform(Xf)
X = X.reshape(-1, 134, 150)
yf = np.zeros(len(yf), dtype=int)

for i in range(0, len(yf), 20):
    yf[i:i+8] = 1


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, yf, test_size=0.2,stratify=yf)

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from keras.layers import Input, Reshape

rl = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rf = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
knn = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
svm = SVC( kernel='rbf', max_iter=10000)
ensamble = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf)], voting='hard')

def model2():
    model = Sequential()
    model.add(keras.layers.InputLayer(shape=(134, 150)))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(64, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(32))
    model.add(Dropout(0.5))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    aja=Dense(1, activation='sigmoid')
    model.add(aja)

    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model


def model3():
    model = Sequential()
    model.add(keras.layers.InputLayer(shape=(134, 8)))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(64, return_sequences=True))
    model.add(Dropout(0.5))
    model.add(LSTM(32))
    model.add(Dropout(0.5))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    aja=Dense(1, activation='sigmoid')
    model.add(aja)

    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model




In [ ]:
np.sum(y_test)/len(y_test)

In [ ]:
model = model2()
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, shuffle=True, batch_size=64)

In [ ]:
#matriz de confusion del modelo
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

y_pred =np.round(model.predict(X_test))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión del modelo')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()

In [ ]:
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)

In [ ]:
svm.fit(Xf_train, yf_train)
knn.fit(Xf_train, yf_train)
rf.fit(Xf_train, yf_train)
rl.fit(Xf_train, yf_train)
ensamble.fit(Xf_train, yf_train)

In [ ]:
print("SVM: ", svm.score(Xf_test, yf_test))
print("kNN: ", knn.score(Xf_test, yf_test))
print("Random Forest: ", rf.score(Xf_test, yf_test))
print("Regresión Logística: ", rl.score(Xf_test, yf_test))
print("Ensamble: ", ensamble.score(Xf_test, yf_test))

In [ ]:
svm.fit(Xs, Ys)
knn.fit(Xs, Ys)
rf.fit(Xs, Ys)
rl.fit(Xs, Ys)
ensamble.fit(Xs, Ys)

In [ ]:
#acuracy de los modelos
print("SVM: ", svm.score(Xsprueba, Ysprueba))
print("kNN: ", knn.score(Xsprueba, Ysprueba))
print("Random Forest: ", rf.score(Xsprueba, Ysprueba))
print("Regresion Logistica: ", rl.score(Xsprueba, Ysprueba))
print("Ensamble: ", ensamble.score(Xsprueba, Ysprueba))

In [ ]:
#clusterin de los datos

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


kmeans = KMeans(n_clusters=2)
kmeans.fit(Xf)
y_kmeans = kmeans.predict(Xf)
plt.scatter(Xf[:, 1000], Xf[:, 2000], c=y_kmeans, s=50, cmap='viridis')
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.5)
plt.show()
np.sum(yf==y_kmeans)/len(yf)

In [ ]:
np.sum(yf==y_kmeans)/len(yf)

In [ ]:
from sklearn.cluster import MeanShift

ms = MeanShift(max_iter=400,)
ms.fit(Xs)


In [ ]:

y_ms = ms.predict(Xf)
plt.scatter(Xf[:, 1000], Xf[:, 2000], c=, s=50, cmap='magma')


In [ ]:
ms.predict(Xf)==yf

In [ ]:
Xf.shape

# PCA


In [ ]:
# se realiza una reduccion de dimensionalidad con PCA

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
pca = PCA(n_components=500)
pca.fit(Xf)
X_pca = pca.transform(Xf)
print("original shape:   ", Xf.shape)
print("transformed shape:", X_pca.shape)
scaler = StandardScaler()
X_pca = scaler.fit_transform(X_pca)


In [ ]:
#se muestra la variancia explicada por cada componente
sum(pca.explained_variance_ratio_)

In [ ]:
# se realiza la gráfica 3D
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(15,10), dpi=2048, facecolor='w', edgecolor='k')

ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=yf, cmap='viridis', s=30)
ax.set_xlabel('component 1', fontsize=20)
ax.set_ylabel('component 2', fontsize=20)
ax.set_zlabel('component 3', fontsize=20)
ax.legend(handles=[mpatches.Patch(color='yellow', label='Deaf'), mpatches.Patch(color='purple', label='Interpreter')], loc='best', fontsize=20)
plt.tight_layout()

# Guardar la figura con mayor resolución
plt.savefig('PCA SI.png', dpi=100)
plt.show()

In [ ]:
#se guarda la grafica
fig.savefig('PCA SI.png', dpi=2048)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure( figsize=(10,10),dpi=1024,layout='constrained')

ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=yf, cmap='viridis', s=50)
ax.set_xlabel('componente 1',fontsize=10)
ax.set_ylabel('componente 2',fontsize=10)
ax.set_zlabel('componente 3',fontsize=10)
ax.set_title('PCA Sordos-Interpretes',pad=0.1,loc='center',fontsize=20)
#color bar
cbar = plt.colorbar(ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=yf, cmap='viridis', s=50))  
plt.show()

# Entrenamiento con caracteristicas PCA

## Sordos - Interprete

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, yf, test_size=0.2,stratify=yf)
rl = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rf = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
knn = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
svm = SVC( kernel='rbf', max_iter=10000)
ensamble = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf),('rl',rl)], voting='hard')
dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')

rl.fit(X_train, y_train)
rf.fit(X_train, y_train)
knn.fit(X_train, y_train)
svm.fit(X_train, y_train)
ensamble.fit(X_train, y_train)
dt.fit(X_train, y_train)

print("RL: ", rl.score(X_test, y_test))
print("RF: ", rf.score(X_test, y_test))
print("kNN: ", knn.score(X_test, y_test))
print("SVM: ", svm.score(X_test, y_test))
print("Ensamble: ", ensamble.score(X_test, y_test))
print("DT: ", dt.score(X_test, y_test))



## Senas

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, yf, test_size=0.2,stratify=yf)
rf = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
knn = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
svm = SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)
ensamble = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf)], voting='hard')
rl = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')

rl.fit(Xf_train, yf_train)
rf.fit(Xf_train, yf_train)
knn.fit(Xf_train, yf_train)
svm.fit(Xf_train, yf_train)
ensamble.fit(Xf_train, yf_train)
dt.fit(Xf_train, yf_train)

print("RL: ", rl.score(Xf_test, yf_test))
print("RF: ", rf.score(Xf_test, yf_test))
print("kNN: ", knn.score(Xf_test, yf_test))
print("SVM: ", svm.score(Xf_test, yf_test))
print("Ensamble: ", ensamble.score(Xf_test, yf_test))
print("DT: ", dt.score(Xf_test, yf_test))


# Importancia de Caracteristicas para S-I


In [ ]:
# se crea un modelo random forest para ver los features mas importantes
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)

dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')
dt.fit(Xf_train, yf_train)


In [ ]:

#importances = dt.feature_importances_
#indices = np.argsort(importances)[::-1]
import matplotlib.pyplot as plt
importances = dt.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 10))
plt.bar(range(Xf_train.shape[1]), importances[indices], align="center")
plt.xticks(range(Xf_train.shape[1]), indices)
plt.ylabel('Importancia', fontsize=20)
plt.xlabel('Características', fontsize=20)
plt.xlim([0, 5])
plt.show()


In [ ]:
sum(importances[indices][:4])

In [ ]:
# ind=[]
# for i in range(100*60*8):
#     Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)
#     dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')
#     dt.fit(Xf_train, yf_train)
#     importances = dt.feature_importances_
#     indices = np.argsort(importances)[::-1]
#     ind.append(indices[0])
# Se lee el archivo de los features
ind = pd.read_csv("featuresimportancia.csv",header=None)
ind = ind.values
ind = ind.reshape(-1,1)
ind = [i[0] for i in ind]

In [ ]:
#ind=[127,145,27,149,41,147,115,141,31,197,81,139,27,29,115,149,3,141,27,137,5,11,148,29,149,35,135,127,141,135,81,39,148,5,69,29,109,65,141,67,136,123,145,1,141,137,73,137,7,35,134,115,137,39,149,123,149]

# se encuentra los 3 valores que mas se repiten

from collections import Counter
contador = Counter(ind).most_common()


In [ ]:
print(x.columns[13268])
print(x.columns[156*38])
print(df2.columns[38])

In [ ]:
aja=[]
for i in range(67):
    aja.append(x.columns[contador[i][0]].split("_")[0])
    print(x.columns[contador[i][0]].split("_")[0])

In [ ]:
# se saca el promedio de la segunda columna de la lista de tuplas

np.mean([x[0] for x in contador[:139]])



In [ ]:
acum=0
i=0
while acum<0.9:
   acum+= contador[i][1]/len(ind)
   i+=1
print(i)

In [ ]:
#se crea histograma de los features mas importantes
#indices menores a 6300 pertenecen a los landmarks de la mano izquierda

import numpy as np

ind_array = np.array(ind)
indmanoizaqquierda = ind_array[ind_array < 6300]
indmanoderecha = ind_array[(ind_array >= 6300) & (ind_array < 12600)]
indcuerpo = ind_array[ind_array >= 12600]
plt.hist(ind_array, bins=len(contador))
plt.title('Histograma de los features')
plt.xlabel('feature')
plt.ylabel('Frecuencia')
plt.show()

plt.hist(np.round(indmanoizaqquierda/(156*2),0), bins=21)
plt.title('Histograma de los features mas importantes de la mano izquierda')
plt.xlabel('Landmark')
plt.ylabel('Frecuencia')
plt.show()

plt.hist(np.round(indmanoderecha/(156*2)-21,0), bins=21)
plt.title('Histograma de los features mas importantes de la mano derecha')
plt.xlabel('Landmark')
plt.ylabel('Frecuencia')
plt.show()

plt.hist(np.round(indcuerpo/(156*2)-21-21,0), bins=33)
plt.title('Histograma de los features mas importantes del cuerpo')
plt.xlabel('Landmark')
plt.ylabel('Frecuencia')
plt.show()


In [ ]:
# Se crea un histograma de los features mas importantes con un color diferente para cada parte del cuerpo
import numpy as np

ind_array = np.array(ind)
indmanoizaqquierda = ind_array[ind_array < 6300]
indmanoderecha = ind_array[(ind_array >= 6300) & (ind_array < 12600)]
indcuerpo = ind_array[ind_array >= 12600]
plt.hist(indmanoizaqquierda, bins=21, color='r')
plt.hist(indmanoderecha, bins=21, color='g')
plt.hist(indcuerpo, bins=33, color='b')
plt.title('Histograma de las características más importantes')
plt.xlabel('Característica')
plt.ylabel('Frecuencia')
plt.legend(['Mano izquierda', 'Mano derecha', 'Cuerpo y cara'])
plt.savefig('PCA SI.png', dpi=100)
plt.show()


In [ ]:
aja=[]
for i in range(67):
    aja.append(x.columns[contador[i][0]].split("_")[1])
Counter(aja).most_common()


In [ ]:
x.columns[12488]

In [ ]:
#se grafica los 3 features mas importantes

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure( figsize=(12,10),dpi=100,facecolor='w',edgecolor='k')
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x[x.columns[13268]], x[x.columns[12488]], x[x.columns[12488]], c=yf, cmap='viridis', s=50)
ax.set_xlabel('RH_Landmark_18'+x.columns[13268][2:],fontsize=17)
ax.set_ylabel('RH_Landmark_19'+x.columns[12488][2:],fontsize=17)
ax.set_zlabel('BD_Landmark_0'+x.columns[12488][2:],fontsize=17)
ax.legend(handles=[mpatches.Patch(color='yellow', label='Sordos'), mpatches.Patch(color='purple', label='Interpretes')], loc='best',fontsize=20)
plt.tight_layout()
plt.savefig('SI3D.png', dpi=100)
plt.show()



# Clasificación S-I con los Landmarks y caracteristicas más importantes

In [ ]:
# se crea un nuevo data set con las columnas 'RH_landmark_18_x','RH_landmark_18_y','RH_landmark_19_x','RH_landmark_19_y','RH_landmark_20_x','RH_landmark_20_y','BD_landmark_0_x','BD_landmark_0_y'

Xi = df2[['RH_landmark_18_x','RH_landmark_18_y','RH_landmark_19_x','RH_landmark_19_y','RH_landmark_20_x','RH_landmark_20_y','BD_landmark_0_x','BD_landmark_0_y']]
Xi = Xi.values
Xi = Xi.reshape(1000,-1,8)
Xfi = x[['36_ECDF Percentile_1','37_ECDF Percentile_1','37_ECDF Percentile_1','38_ECDF Percentile_1','39_ECDF Percentile_1','40_ECDF Percentile_1','41_ECDF Percentile_1','42_ECDF Percentile_1']]


In [ ]:


Xi_train, Xi_test, yf_train, yf_test = train_test_split(Xi, yf, test_size=0.2,stratify=yf)
Xfi_train, Xfi_test, yfi_train, yfi_test = train_test_split(Xfi, yf, test_size=0.2,stratify=yf)



rl = LogisticRegression(max_iter=10000, C=0.1, solver='liblinear', penalty='l2', tol=0.0001)
rf = RandomForestClassifier(max_depth=9, min_samples_leaf=1, min_samples_split=4, n_estimators=200)
knn = KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')
svm = SVC( kernel='rbf', max_iter=10000)
ensamble = VotingClassifier(estimators=[('svm', svm), ('knn', knn), ('rf', rf),('rl',rl)], voting='hard')
dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')

rl.fit(Xfi_train, yfi_train)
rf.fit(Xfi_train, yfi_train)
knn.fit(Xfi_train, yfi_train)
svm.fit(Xfi_train, yfi_train)
ensamble.fit(Xfi_train, yfi_train)
dt.fit(Xfi_train, yfi_train)

print("RL: ", rl.score(Xfi_test, yfi_test))
print("RF: ", rf.score(Xfi_test, yfi_test))
print("kNN: ", knn.score(Xfi_test, yfi_test))
print("SVM: ", svm.score(Xfi_test, yfi_test))
print("Ensamble: ", ensamble.score(Xfi_test, yfi_test))
print("DT: ", dt.score(Xfi_test, yfi_test))



# Caracteristicas mas importantes Clasificación de señas

In [ ]:
# Se dividen los datos en caracteristicas y etiquetas

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xf = carac.iloc[:, :-1].values
yf = carac["etiqueta"].values[:]

scaler = StandardScaler()
Xf = scaler.fit_transform(Xf)


ind2 = []

for i in range(20):

    Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2,stratify=yf)
    dt = DecisionTreeClassifier(max_depth=9, min_samples_leaf=2,min_samples_split=2,splitter='random',criterion='entropy')
    dt.fit(Xf_train, yf_train)
    importances = dt.feature_importances_
    indices = np.argsort(importances)[::-1]
    ind2.append(indices[:169])

In [ ]:
acum=0
i=0
while acum<0.99:
   acum+= importances[indices[i]]
   i+=1
print(i)
print(len(importances))

In [ ]:
ind2

In [ ]:
frames=np.linspace(1, 134, 134)

In [ ]:
#se grafican landmarks

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig = plt.figure( figsize=(10,10))

plt.plot(frames,df2['RH_landmark_18_x'][:134],'r')
plt.plot(frames,df2['RH_landmark_19_x'][:134],'g')
plt.plot(frames,df2['RH_landmark_20_x'][:134],'b')

plt.title('Marcadores anatomicos de intérprete')
plt.xlabel('Frame')
plt.ylabel('Posición en x')
plt.legend(['RH_landmark_18','RH_landmark_19','RH_landmark_20'])
plt.show()
fig = plt.figure( figsize=(10,10))
plt.plot(frames,df2['RH_landmark_18_x'][134*8:134*9],'r')
plt.plot(frames,df2['RH_landmark_19_x'][134*8:134*9],'g')
plt.plot(frames,df2['RH_landmark_20_x'][134*8:134*9],'b')
plt.title('Marcadores anatomicos de sordo')
plt.xlabel('Frame')
plt.ylabel('Posición en x')
plt.legend(['RH_landmark_18','RH_landmark_19','RH_landmark_20'])
plt.show()




# nuevo

In [ ]:
# Análisis de Landmarks para LSC: Métricas y Visualizaciones
# ===========================================================

# Importación de bibliotecas necesarias
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tsfel
# Configuración estética para gráficas
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 14
colors = sns.color_palette('viridis', 5)


In [ ]:
# Visualización de PCA para los features
def visualizar_pca(X_features, y_binary, n_components=3):
    """
    Realiza PCA y visualiza los datos en 3D coloreados por tipo de usuario.
    """
    print("Realizando análisis de componentes principales...")
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_features)
    
    # Calcular varianza explicada
    explained_variance = pca.explained_variance_ratio_
    print(f"Varianza explicada por componente: {explained_variance}")
    print(f"Varianza explicada acumulada: {np.sum(explained_variance):.4f}")
    
    # Gráfico 3D de los componentes principales
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], 
               c=y_binary, cmap='viridis', s=50, alpha=0.7)
    
    # Añadir leyenda, etiquetas y título
    legend1 = ax.legend(*scatter.legend_elements(),
                        title="Usuarios")
    labels = ['Sordos', 'Intérpretes']
    handles, _ = scatter.legend_elements()
    ax.legend([handles[0], handles[1]], labels, loc="upper right", title="Tipo de Usuario")
    
    ax.set_xlabel(f'Componente 1 ({explained_variance[0]:.2%})', fontsize=12)
    ax.set_ylabel(f'Componente 2 ({explained_variance[1]:.2%})', fontsize=12)
    ax.set_zlabel(f'Componente 3 ({explained_variance[2]:.2%})', fontsize=12)
    ax.set_title('Visualización PCA de Características de Landmarks', fontsize=16)
    
    plt.tight_layout()
    plt.savefig('pca_landmarks_3d.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return X_pca, pca

# Ejecutar PCA y visualizar
X_pca, pca_model = visualizar_pca(X_features, y_binary)

# 3. Modelos de Clasificación
# ==========================

# Configurar protocolos de evaluación
def configurar_evaluacion():
    """
    Configura los diferentes protocolos de evaluación.
    """
    # Protocolo 1: Partición aleatoria estratificada (80% train, 20% test)
    X_train1, X_test1, y_train1, y_test1 = train_test_split(
        X_features, y, test_size=0.2, stratify=y, random_state=42)
    
    X_train1_binary, X_test1_binary, y_train1_binary, y_test1_binary = train_test_split(
        X_features, y_binary, test_size=0.2, stratify=y_binary, random_state=42)
    
    # Protocolo 2: Leave-one-repetition-out
    # Implementación simplificada - se reservan samples específicos para validación
    X_train2, X_test2, y_train2, y_test2 = [], [], [], []
    X_train2_binary, X_test2_binary, y_train2_binary, y_test2_binary = [], [], [], []
    
    # Para simplificar, asumimos que cada 4 muestras pertenecen a una repetición
    for i in range(0, len(y), 4):
        # Reservar una muestra para test
        mask = np.ones(len(y), dtype=bool)
        mask[i:i+1] = False  # Una muestra para test
        
        X_train2.append(X_features[mask])
        X_test2.append(X_features[~mask])
        y_train2.append(y[mask])
        y_test2.append(y[~mask])
        
        X_train2_binary.append(X_features[mask])
        X_test2_binary.append(X_features[~mask])
        y_train2_binary.append(y_binary[mask])
        y_test2_binary.append(y_binary[~mask])
    
    # Protocolo 3: Leave-one-subject-out
    # Implementación para 5 sujetos - se entrena con 4 y se evalúa con 1
    X_per_subject = {}
    y_per_subject = {}
    X_per_subject_binary = {}
    y_per_subject_binary = {}
    
    # Asumimos 5 voluntarios, cada uno con 200 muestras (1000/5)
    for i in range(5):
        start_idx = i * 200
        end_idx = (i + 1) * 200
        
        X_per_subject[i] = X_features[start_idx:end_idx]
        y_per_subject[i] = y[start_idx:end_idx]
        X_per_subject_binary[i] = X_features[start_idx:end_idx]
        y_per_subject_binary[i] = y_binary[start_idx:end_idx]
    
    return {
        'random_stratified': {
            'sign': (X_train1, X_test1, y_train1, y_test1),
            'binary': (X_train1_binary, X_test1_binary, y_train1_binary, y_test1_binary)
        },
        'leave_one_repetition': {
            'sign': (X_train2, X_test2, y_train2, y_test2),
            'binary': (X_train2_binary, X_test2_binary, y_train2_binary, y_test2_binary)
        },
        'leave_one_subject': {
            'sign': (X_per_subject, y_per_subject),
            'binary': (X_per_subject_binary, y_per_subject_binary)
        }
    }

# Entrenar y evaluar modelos
def entrenar_evaluar_modelos(protocolos):
    """
    Entrena y evalúa múltiples modelos según los protocolos establecidos.
    """
    # Definir modelos a entrenar
    models = {
        'SVM': SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000),
        'KNN': KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance'),
        'Random Forest': RandomForestClassifier(max_depth=9, min_samples_leaf=1, 
                                              min_samples_split=4, n_estimators=200),
        'Ensemble': VotingClassifier(estimators=[
            ('svm', SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)), 
            ('knn', KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')), 
            ('rf', RandomForestClassifier(max_depth=9, min_samples_leaf=1, 
                                        min_samples_split=4, n_estimators=200))
        ], voting='hard')
    }
    
    results = {
        'random_stratified': {
            'sign': {},
            'binary': {}
        },
        'leave_one_repetition': {
            'sign': {},
            'binary': {}
        },
        'leave_one_subject': {
            'sign': {},
            'binary': {}
        }
    }
    
    # 1. Protocolo de partición aleatoria estratificada
    print("\nEntrenando modelos con partición aleatoria estratificada...")
    X_train, X_test, y_train, y_test = protocolos['random_stratified']['sign']
    X_train_binary, X_test_binary, y_train_binary, y_test_binary = protocolos['random_stratified']['binary']
    
    for name, model in models.items():
        print(f"  Entrenando {name} para clasificación de señas...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        cm = confusion_matrix(y_test, y_pred)
        
        results['random_stratified']['sign'][name] = {
            'accuracy': accuracy,
            'report': report,
            'confusion_matrix': cm,
            'model': model
        }
        
        print(f"  Entrenando {name} para clasificación sordo-intérprete...")
        model_binary = models[name].__class__()
        if name == 'Ensemble':
            model_binary = VotingClassifier(estimators=[
                ('svm', SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)), 
                ('knn', KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')), 
                ('rf', RandomForestClassifier(max_depth=9, min_samples_leaf=1, 
                                          min_samples_split=4, n_estimators=200))
            ], voting='hard')
        
        model_binary.fit(X_train_binary, y_train_binary)
        y_pred_binary = model_binary.predict(X_test_binary)
        accuracy_binary = accuracy_score(y_test_binary, y_pred_binary)
        report_binary = classification_report(y_test_binary, y_pred_binary, output_dict=True)
        cm_binary = confusion_matrix(y_test_binary, y_pred_binary)
        
        results['random_stratified']['binary'][name] = {
            'accuracy': accuracy_binary,
            'report': report_binary,
            'confusion_matrix': cm_binary,
            'model': model_binary
        }
    
    # 2. Implementación simplificada del Protocolo leave-one-repetition-out
    # Para una implementación completa, se requeriría un ajuste según la estructura específica de los datos
    print("\nImplementando evaluación simplificada para protocolo leave-one-repetition-out...")
    
    # 3. Protocolo leave-one-subject-out
    print("\nEntrenando modelos con protocolo leave-one-subject-out...")
    X_per_subject, y_per_subject = protocolos['leave_one_subject']['sign']
    X_per_subject_binary, y_per_subject_binary = protocolos['leave_one_subject']['binary']
    
    for subject_out in range(5):
        results['leave_one_subject']['sign'][f'Subject_{subject_out+1}'] = {}
        results['leave_one_subject']['binary'][f'Subject_{subject_out+1}'] = {}
        
        # Crear conjuntos de train y test
        X_train = np.vstack([X_per_subject[i] for i in range(5) if i != subject_out])
        y_train = np.concatenate([y_per_subject[i] for i in range(5) if i != subject_out])
        X_test = X_per_subject[subject_out]
        y_test = y_per_subject[subject_out]
        
        X_train_binary = np.vstack([X_per_subject_binary[i] for i in range(5) if i != subject_out])
        y_train_binary = np.concatenate([y_per_subject_binary[i] for i in range(5) if i != subject_out])
        X_test_binary = X_per_subject_binary[subject_out]
        y_test_binary = y_per_subject_binary[subject_out]
        
        for name, model in models.items():
            print(f"  Entrenando {name} para sujeto {subject_out+1} (clasificación de señas)...")
            model_copy = models[name].__class__()
            if name == 'Ensemble':
                model_copy = VotingClassifier(estimators=[
                    ('svm', SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)), 
                    ('knn', KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')), 
                    ('rf', RandomForestClassifier(max_depth=9, min_samples_leaf=1, 
                                              min_samples_split=4, n_estimators=200))
                ], voting='hard')
            
            model_copy.fit(X_train, y_train)
            y_pred = model_copy.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            
            results['leave_one_subject']['sign'][f'Subject_{subject_out+1}'][name] = {
                'accuracy': accuracy,
                'model': model_copy
            }
            
            print(f"  Entrenando {name} para sujeto {subject_out+1} (clasificación sordo-intérprete)...")
            model_binary = models[name].__class__()
            if name == 'Ensemble':
                model_binary = VotingClassifier(estimators=[
                    ('svm', SVC(C=10, gamma=0.0001, kernel='rbf', max_iter=10000)), 
                    ('knn', KNeighborsClassifier(algorithm='auto', n_neighbors=3, weights='distance')), 
                    ('rf', RandomForestClassifier(max_depth=9, min_samples_leaf=1, 
                                              min_samples_split=4, n_estimators=200))
                ], voting='hard')
            
            model_binary.fit(X_train_binary, y_train_binary)
            y_pred_binary = model_binary.predict(X_test_binary)
            accuracy_binary = accuracy_score(y_test_binary, y_pred_binary)
            
            results['leave_one_subject']['binary'][f'Subject_{subject_out+1}'][name] = {
                'accuracy': accuracy_binary,
                'model': model_binary
            }
    
    return results

# 4. Modelo LSTM para landmarks secuenciales
# =========================================

def entrenar_lstm(X, y_binary):
    """
    Entrena un modelo LSTM para clasificación de tipo de usuario usando landmarks secuenciales.
    """
    print("\nPreparando datos para modelo LSTM...")
    
    # Dividir datos
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.2, stratify=y_binary, random_state=42)
    
    # Definir modelo LSTM
    print("Construyendo modelo LSTM...")
    model = Sequential([
        LSTM(32, return_sequences=True, input_shape=(134, 150)),
        Dropout(0.5),
        LSTM(64, return_sequences=True),
        Dropout(0.5),
        LSTM(128),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    # Compilar modelo
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Callback para early stopping
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True
    )
    
    # Entrenar modelo
    print("Entrenando modelo LSTM...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,  # Reducido para este ejemplo, aumentar para mejor rendimiento
        batch_size=64,
        callbacks=[early_stopping]
    )
    
    # Evaluar modelo
    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"LSTM Accuracy: {accuracy:.4f}")
    
    # Graficar curva de aprendizaje
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Precisión del Modelo')
    plt.ylabel('Precisión')
    plt.xlabel('Época')
    plt.legend(['Entrenamiento', 'Validación'], loc='lower right')
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Pérdida del Modelo')
    plt.ylabel('Pérdida')
    plt.xlabel('Época')
    plt.legend(['Entrenamiento', 'Validación'], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('lstm_learning_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Visualizar matriz de confusión
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Sordo', 'Intérprete'], 
                yticklabels=['Sordo', 'Intérprete'])
    plt.title('Matriz de Confusión LSTM')
    plt.xlabel('Clase Predicha')
    plt.ylabel('Clase Real')
    plt.savefig('lstm_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'model': model,
        'history': history,
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': cm
    }

# 5. Análisis de Importancia de Características
# ===========================================

def analisis_importancia(X_features, y_binary):
    """
    Analiza la importancia de las características usando Random Forest.
    """
    print("\nAnalizando importancia de características...")
    
    # Entrenar Random Forest
    rf = RandomForestClassifier(
        max_depth=9, 
        min_samples_leaf=1, 
        min_samples_split=4, 
        n_estimators=200,
        random_state=42
    )
    
    rf.fit(X_features, y_binary)
    
    # Obtener importancia de características
    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    # Graficar las 20 características más importantes
    plt.figure(figsize=(12, 8))
    plt.bar(range(20), importances[indices[:20]], align="center")
    plt.xticks(range(20), indices[:20], rotation=90)
    plt.xlabel('Índice de la Característica')
    plt.ylabel('Importancia')
    plt.title('Importancia de Características para Clasificación Sordo-Intérprete')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Identificar categorías de las características más importantes
    # Aquí simulamos la identificación de categorías anatómicas
    # En un análisis real, esto requeriría mapear índices a nombres de landmarks
    
    feature_categories = {
        'Mano derecha': [],
        'Mano izquierda': [],
        'Cuerpo y cara': []
    }
    
    # Clasificación simplificada basada en índices (ajustar según estructura real)
    for idx in indices[:100]:  # Analizar las 100 más importantes
        if idx < 6300:
            feature_categories['Mano izquierda'].append(idx)
        elif idx < 12600:
            feature_categories['Mano derecha'].append(idx)
        else:
            feature_categories['Cuerpo y cara'].append(idx)
    
    # Graficar distribución por categoría
    category_counts = [len(v) for v in feature_categories.values()]
    
    plt.figure(figsize=(10, 6))
    plt.bar(feature_categories.keys(), category_counts, color=colors[:3])
    plt.title('Distribución de Características Importantes por Región Anatómica')
    plt.ylabel('Número de Características')
    plt.savefig('feature_importance_by_category.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'importances': importances,
        'indices': indices,
        'feature_categories': feature_categories
    }

# 6. Visualizaciones de Resultados
# ==============================

def visualizar_resultados(resultados):
    """
    Genera visualizaciones para los resultados de los modelos.
    """
    # 1. Comparación de precisión para clasificación de señas
    modelos = list(resultados['random_stratified']['sign'].keys())
    acc_random = [resultados['random_stratified']['sign'][m]['accuracy'] for m in modelos]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(modelos, acc_random, color=colors[:len(modelos)])
    
    plt.title('Precisión de Modelos para Clasificación de Señas (Partición Aleatoria)')
    plt.ylabel('Precisión')
    plt.ylim(0, 1.0)
    
    # Añadir etiquetas de valor
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('model_comparison_signs.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Comparación de precisión para clasificación sordo-intérprete
    acc_binary = [resultados['random_stratified']['binary'][m]['accuracy'] for m in modelos]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(modelos, acc_binary, color=colors[:len(modelos)])
    
    plt.title('Precisión de Modelos para Clasificación Sordo-Intérprete (Partición Aleatoria)')
    plt.ylabel('Precisión')
    plt.ylim(0.5, 1.0)  # Ajustado para destacar diferencias
    
    # Añadir etiquetas de valor
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('model_comparison_binary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 3. Matriz de confusión para el mejor modelo en clasificación de señas
    best_model_name = modelos[np.argmax(acc_random)]
    cm = resultados['random_stratified']['sign'][best_model_name]['confusion_matrix']
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusión - {best_model_name} (Clasificación de Señas)')
    plt.xlabel('Clase Predicha')
    plt.ylabel('Clase Real')
    plt.tight_layout()
    plt.savefig('confusion_matrix_signs.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 4. Comparación de leave-one-subject-out
    # Preparar datos para gráfico
    subjects = [f'Subject_{i+1}' for i in range(5)]
    model_acc = {model: [] for model in modelos}
    
    for subject in subjects:
        for model in modelos:
            acc = resultados['leave_one_subject']['sign'][subject][model]['accuracy']
            model_acc[model].append(acc)
    
    # Gráfico de barras agrupadas
    fig, ax = plt.subplots(figsize=(14, 8))
    bar_width = 0.15
    index = np.arange(len(subjects))
    
    for i, model in enumerate(modelos):
        plt.bar(index + i*bar_width, model_acc[model], bar_width,
                label=model, color=colors[i])
    
    plt.xlabel('Sujeto Excluido')
    plt.ylabel('Precisión')
    plt.title('Precisión de Modelos con Protocolo Leave-One-Subject-Out')
    plt.xticks(index + bar_width * (len(modelos)-1)/2, subjects)
    plt.legend()
    plt.tight_layout()
    plt.savefig('leave_one_subject_out.png', dpi=300, bbox_inches='tight')
    plt.show()

# Ejecutar todo el análisis
# =========================

# Configurar protocolos de evaluación
print("Configurando protocolos de evaluación...")
protocolos = configurar_evaluacion()

# Entrenar y evaluar modelos
print("Entrenando y evaluando modelos clásicos...")
resultados = entrenar_evaluar_modelos(protocolos)

# Entrenar modelo LSTM
print("Entrenando modelo LSTM para datos secuenciales...")
resultados_lstm = entrenar_lstm(X, y_binary)

# Analizar importancia de características
print("Analizando importancia de características...")
importancia = analisis_importancia(X_features, y_binary)

# Visualizar resultados
print("Generando visualizaciones de resultados...")
visualizar_resultados(resultados)

print("\nAnálisis completo. Todas las gráficas han sido guardadas.")